# 05.3 — A bounded harness

[Class index and setup](README.md) · [Previous: 05.2](05.2-two-read-only-tools.ipynb) · [Next: 05.4](05.4-guided-failure-practice.ipynb)

**Purpose:** Read the supplied loop through its observable trace.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's project is additional.

**Starting knowledge:** Basic Python functions, dictionaries, JSON/schema validation, and the earlier pytest ideas. All code and saved inputs are supplied again; no earlier exercise answers or paid key are needed.

**Learning outcomes:** Count requests versus executions; recognize stopped outcomes; complete a one-number test assertion.

[Glossary](GLOSSARY.md) · [Supplied code](report_app/__init__.py) · [Fixture provenance](fixtures/README.md)

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/harness.py").is_file() or not Path("fixtures/world-bank-population.json").is_file():
    raise FileNotFoundError(
        "Open 05-small-ai-applications.code-workspace and restart the kernel. "
        "Download the complete class folder with report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available. Default execution is REPLAY, with no HTTP or credentials.")

## In class · 60–68 minutes — The loop is supplied

The harness is ordinary Python around one callable named `request`. In replay, that callable returns a saved response. In live mode it makes a direct SDK request. Tools, argument validation, output validation, trace, and counters are shared application code. There is no provider hierarchy or orchestration framework.

A successful lookup normally takes **two model requests**: first propose the lookup, then answer using its result. Only **one Python tool** executes. A comparison can use two lookups and a third request for the answer. An ordinary explanation that writing is unavailable can finish with zero tools. Counters record what actually happened, not an assumed plan.

```text
Before request: check request budget
Request response: inspect completion/refusal
If tool proposed: check execution budget → validate → execute → append result
If final text: validate Answer → finish
Any failure: finish with a named outcome
```

Python permits at most **three model requests** and **two tool executions** per run. A third proposed tool is visible in the trace but never executes. These custom-function limits are enforced by the application, not by an SDK parameter. One tool proposal is processed at a time; multiple simultaneous proposals are rejected.

In [ ]:
from report_app.harness import run, MAX_MODEL_REQUESTS, MAX_TOOL_EXECUTIONS
from report_app.provider import make_replay
from report_app.source import read_fixture

script = read_fixture("replays/portugal.json")
lookup_run = run(script["prompt"], make_replay("portugal"))
print("Mode:", lookup_run["mode"], "Outcome:", lookup_run["outcome"])
print("Requests:", lookup_run["model_requests"], "Tools:", lookup_run["tool_executions"])
for event in lookup_run["trace"]:
    print(event)
assert (MAX_MODEL_REQUESTS, MAX_TOOL_EXECUTIONS) == (3, 2)

## In class · 68–80 minutes — An observable stop is useful

A trace records application events: request number, proposal name/arguments, actual tool result, and final outcome. It does not expose private model reasoning. The supplied code preserves full provider output in conversation history when needed for a subsequent request, while keeping the displayed trace focused on observable events.

The next saved script keeps proposing `get_summary`. The third proposal must stop the run. No fourth request is sent, no third function executes, and no unsupported answer is presented as success. Compare proposed calls and executed results rather than relying only on the final counter.

In [ ]:
repeated_script = read_fixture("replays/repeated.json")
stopped = run(repeated_script["prompt"], make_replay("repeated"))
print("REPLAY:", stopped["outcome"], stopped["detail"])
for event in stopped["trace"]:
    print(event)
print("Proposals:", sum(e["event"] == "tool_proposed" for e in stopped["trace"]))
print("Results:", sum(e["event"] == "tool_result" for e in stopped["trace"]))

### Tests and evaluation ask different questions

A deterministic test supplies a known proposal and checks application behavior: can a forbidden name execute; can a third tool run; does a malformed answer get accepted? It should be repeatable and independent of provider availability.

Evaluation asks whether the application behaves usefully for a prompt: does it report the right population, compare correctly, or explain missing data? We supply five prompts and explicit expectations. Through replay, their results demonstrate the authored scenario and application flow. Live evaluation adds evidence about the selected model at that time; it may vary across runs. A five-prompt set is a small diagnostic exercise, not a broad quality guarantee.

Schema-valid text can still contain a wrong number. Always inspect numeric claims against the actual tool results. Passing the stopping test establishes a limit; it does not establish factual accuracy.

```python
script = read_fixture("replays/repeated.json")
result = run(script["prompt"], make_replay("repeated"))
assert result["outcome"] == "tool_limit"
assert result["tool_executions"] == ____
assert result["model_requests"] == 3
```

The supplied test suite also spies on actual dispatch calls, so an incorrect counter alone cannot hide an extra execution. You do not implement the loop.

### Exercise 05.3.E1 — Assert the execution limit

> **In class · 10 minutes · 80–90**
>
> **Goal:** Complete one assertion and explain the third proposal.
>
> **Start here:** Copy the small test-shaped snippet below; run and replay are already supplied.
>
> **Task:** Replace `____` with the maximum number of actual tool executions. Run the snippet. Find the third proposal in the trace and explain why it has no corresponding tool result.
>
> **Check:** Outcome tool_limit; three requests; two executions; no fourth request.
>
> **Hint:** The allowed number is `2`; proposals and executions are different events.

In [ ]:
# Your attempt for 05.3.E1.
# Complete only the tiny expression; record observations/explanations here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_05_3_e1.py

## Guided follow-up — Independent work · 15 minutes

Spend five minutes comparing the Portugal trace with the repeated-call trace. Spend five tracing the comparison script (two lookups and one answer). Spend five writing where the application checks each budget before the corresponding action. These minutes complete the three 15-minute classroom-part follow-ups in the guided-walkthrough allocation.

**Completion:** identify both limits and one successful stopping condition. A model refusal and an ordinary answer explaining read-only limits are distinct outcomes.

## References

[Supplied bounded loop](report_app/harness.py) · [Official Responses reference](https://developers.openai.com/api/reference/python/resources/responses/methods/create)